In [1]:
import os
os.chdir('../')

import argparse
import json
from tqdm.notebook import tqdm

from PIL import Image
import torch
import torch.nn as nn
from torchvision.transforms.functional import to_pil_image
from huggingface_hub import hf_hub_download

import timm
from timm.data import resolve_data_config
from timm.data.transforms_factory import create_transform
import pickle

from model import *

In [4]:
model = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)
transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))
# model.eval()

In [5]:
sample = torch.zeros((1,3,448,448))
with torch.inference_mode():
    print(model(sample).shape) #im guessing this is just the CLS token embedding 

torch.Size([1, 1024])


In [6]:
from torchsummary import summary
summary(model, input_size=(3, 224, 224))

ModuleNotFoundError: No module named 'torchsummary'

In [ ]:
from model import ABMIL

mil_batch = torch.zeros((128, 3, 224, 224)) #128 instances of 224x224 rgb

class UniABMIL(nn.Module):
    def __init__(self, encoder_args = None, emb_sz = 1024):
        super().__init__()
        self.encoder = timm.create_model("hf-hub:MahmoodLab/uni", pretrained=True, init_values=1e-5, dynamic_img_size=True)
        self.transform = create_transform(**resolve_data_config(model.pretrained_cfg, model=model))

        #all samples in a bag are projected into hidden_sz space
        #and then those projected embs are weighted summed into
        #a single hidden_sz embedding 
        self.abmil = ABMIL(emb_sz = emb_sz, hidden_sz=1024)

        self.class_head = nn.Sequential(
            nn.Linear(1024, 512),
            nn.LeakyReLU(),
            nn.Linear(512, 4)
        )

    def forward(self, x):
        x = self.transform(x)
        x = self.encoder(x)

        x = self.abmil(x)
        x = self.class_head(x)
        return x

uniabmil = UniABMIL()
uniabmil(mil_batch).shape


torch.Size([4])

In [ ]:
def pad_batch(batch, pad_batch_size):
    orig_len = len(batch)
    if len(batch) != pad_batch_size:
        _ = torch.zeros((pad_batch_size - len(batch),) + batch.shape[1:]).type(batch.type())
        batch = torch.vstack([batch, _])

    return batch, orig_len

In [8]:
from transformers import pipeline

# Load the model and tokenizer
# Model: https://huggingface.co/OpenMed/OpenMed-NER-PathologyDetect-PubMed-109M
model_name = "OpenMed/OpenMed-NER-PathologyDetect-PubMed-109M"

# Create a pipeline
medical_ner_pipeline = pipeline(
    model=model_name,
    # aggregation_strategy="simple",
    task="feature-extraction",
    framework="pt",
    return_tensor=True,
)

# Example usage
text = "Early detection of breast cancer improves survival rates."
entities = medical_ner_pipeline(text)

print(len(entities))


Some weights of BertModel were not initialized from the model checkpoint at OpenMed/OpenMed-NER-PathologyDetect-PubMed-109M and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Device set to use cuda:0


1


In [9]:
entities

[[[0.470703125,
   -0.1689453125,
   -0.1337890625,
   -0.435546875,
   -0.21484375,
   -0.37890625,
   0.12890625,
   -0.0167236328125,
   -0.03662109375,
   0.22265625,
   0.2265625,
   -0.01312255859375,
   -0.17578125,
   -0.2294921875,
   -0.1552734375,
   -0.43359375,
   0.341796875,
   0.162109375,
   0.373046875,
   0.1796875,
   0.10693359375,
   -0.294921875,
   0.0556640625,
   0.24609375,
   0.4921875,
   0.69140625,
   0.1015625,
   0.0888671875,
   0.3359375,
   0.0244140625,
   0.0849609375,
   0.54296875,
   -0.08935546875,
   -0.0228271484375,
   -0.19921875,
   0.126953125,
   -0.1953125,
   -0.1494140625,
   1.515625,
   -0.287109375,
   -0.0634765625,
   0.55078125,
   -0.2265625,
   0.416015625,
   0.15234375,
   0.404296875,
   -0.412109375,
   -0.08203125,
   0.47265625,
   -0.07421875,
   -0.1728515625,
   -0.053955078125,
   0.3515625,
   0.275390625,
   0.33203125,
   -0.1591796875,
   -0.208984375,
   0.2177734375,
   -0.0245361328125,
   -0.11083984375,
   0

In [1]:
!python run_single_slide.py --slide_path D:/pancreatic_cancer/2025-08-19 PANC/PANC1.svs --job_dir ./trident_processed --patch_encoder uni_v1 --mag 20 --patch_size 256

python: can't open file 'c:\\Users\\BrownRadX\\Documents\\panc\\multi\\Notebooks\\run_single_slide.py': [Errno 2] No such file or directory


In [6]:
import h5py
f1 = h5py.File('../../20x_256px_0px_overlap/patches/PANC1_patches.h5', 'r')
f1.keys()

<KeysViewHDF5 ['coords']>

In [7]:
f1['coords']

<HDF5 dataset "coords": shape (23309, 2), type "<i8">

In [8]:
import h5py
f1 = h5py.File('../../20x_256px_0px_overlap/features_uni_v1/PANC1.h5', 'r')
f1.keys()

<KeysViewHDF5 ['coords', 'features']>

In [9]:
f1['coords'], f1['features']

(<HDF5 dataset "coords": shape (23309, 2), type "<i8">,
 <HDF5 dataset "features": shape (23309, 1024), type "<f4">)

In [14]:
len(f1['features'])

23309

In [12]:
import numpy as np
np.array(f1['features'], dtype=np.float16)

array([[-0.5405 , -0.1643 ,  0.6187 , ..., -0.509  ,  1.384  ,  0.322  ],
       [-1.817  , -0.3833 ,  0.853  , ..., -0.3389 , -0.7324 , -0.831  ],
       [-1.423  ,  1.035  ,  1.097  , ..., -0.5513 , -2.057  , -1.647  ],
       ...,
       [-1.589  , -1.46   ,  0.04614, ...,  0.1501 , -1.378  , -2.092  ],
       [-0.4028 , -1.536  ,  0.1372 , ..., -0.1927 , -0.414  , -2.32   ],
       [ 0.0809 ,  0.792  ,  0.7227 , ..., -0.0749 ,  1.19   , -2.436  ]],
      shape=(23309, 1024), dtype=float16)

In [15]:
np.array(f1['coords']).dtype

dtype('int64')

In [7]:
import json
import numpy as np
out = "../../uni_embs"
with open(f"{out}/meta.json") as f:
    obj = json.load(f)

dim = 1024
dtype = "float16"
ids = np.zeros((len(obj)))
offsets = np.zeros((len(obj)))
num_patches = np.zeros((len(obj)))

for ind, (slide, slide_details) in enumerate(obj.items()):
    print(slide, slide_details)
    ids[ind] = int(slide[4:])
    offsets[ind] = slide_details["offset"]
    num_patches[ind] = slide_details["num_patches"]


PANC1 {'num_patches': 23309, 'offset': 0}
PANC10 {'num_patches': 56786, 'offset': 23309}
PANC100 {'num_patches': 41846, 'offset': 80095}
PANC101 {'num_patches': 32132, 'offset': 121941}
PANC103 {'num_patches': 39976, 'offset': 154073}
PANC104 {'num_patches': 19777, 'offset': 194049}
PANC105 {'num_patches': 42880, 'offset': 213826}
PANC106 {'num_patches': 25777, 'offset': 256706}
PANC107 {'num_patches': 30906, 'offset': 282483}
PANC108 {'num_patches': 43355, 'offset': 313389}
PANC109 {'num_patches': 46083, 'offset': 356744}
PANC11 {'num_patches': 51397, 'offset': 402827}
PANC110 {'num_patches': 25337, 'offset': 454224}
PANC111 {'num_patches': 47562, 'offset': 479561}
PANC112 {'num_patches': 46880, 'offset': 527123}
PANC113 {'num_patches': 36156, 'offset': 574003}
PANC114 {'num_patches': 26044, 'offset': 610159}
PANC115 {'num_patches': 35022, 'offset': 636203}
PANC116 {'num_patches': 24292, 'offset': 671225}
PANC117 {'num_patches': 37234, 'offset': 695517}
PANC118 {'num_patches': 49901, 

In [8]:
ids

array([  1.,  10., 100., 101., 103., 104., 105., 106., 107., 108., 109.,
        11., 110., 111., 112., 113., 114., 115., 116., 117., 118., 119.,
        12., 120., 122., 123., 124., 125., 126., 127., 128., 129.,  13.,
       130., 131., 132., 133., 134., 135., 136., 138., 139.,  14., 140.,
       141., 142., 143., 144., 145., 146., 147., 148., 149.,  15., 150.,
       152., 153., 154., 155., 156., 157., 158., 159., 160., 161., 162.,
       163., 164., 165., 166., 167., 168., 169.,  17., 170., 171., 172.,
       173., 174., 175., 176., 177., 178., 179.,  18., 180., 181., 182.,
       183., 184., 185., 186., 187., 188., 189.,  19., 190., 191., 192.,
       193., 194., 195., 196., 198., 199.,   2.,  20., 200., 201., 202.,
       203., 204., 205., 206., 207., 208., 209.,  21., 210., 211., 213.,
       214., 215., 216., 217., 218., 219.,  22., 220., 222., 223., 224.,
       225., 226., 227., 228., 229.,  23., 230.,  24.,  26.,  27.,  28.,
        30.,  31.,  32.,  33.,  34.,  35.,  36.,  3

In [ ]:
np.savez("../../uni_embs/index_arrays.npz", 
         dtype = dtype, 
         feat_dim = dim, 
         offsets = offsets, 
         lengths = num_patches, 
         slide_ids = ids,
         total_patches = 8206640)

In [8]:
index = np.load("../uni_embs/index_arrays.npz", allow_pickle=True)
dtype = index["dtype"]
feat_dim = index["feat_dim"]
offsets = index["offsets"]
lengths = index["lengths"]
ids = index["slide_ids"]
total_patches = index["total_patches"]

In [12]:
import pandas as pd
df = pd.read_excel("pan_clinical_data.xlsx")
df = df[df["Excluded initially"].isna()]
df = df[df["PDAC UniID"].notna()]

#for now, the simplest formulation of this problem is a multimodal
# img + "Path Diagnosis" embs -> overall survival
# I think from an SSL standpoint it makes sense
# for the model to train on reconstruction from joint embedding space first
# and then to use different linear probes 
drop_cols = ["Excluded initially", "Medical Rec #", "Unnamed: 4", \
                "Date of Birth", "Date 1st Recurrence", \
                "Date of Last Contact or Death if Deceased", \
                "Excluded initially", "Medical Rec #",
                "Date of Initial Diagnosis", "Cancer Status"]
df = df.drop(labels=drop_cols, axis=1)
df["PDAC UniID"] = df["PDAC UniID"].astype(int)

In [ ]:
ids = ids.astype(int)
mask = [id in df["PDAC UniID"].values for id in ids]
offsets = offsets[mask]
lengths = lengths[mask]
ids = ids[mask]
total_patches = np.sum(lengths)

In [ ]:
vitals = {"Dead": 0.0, "Alive": 1.0, np.nan: np.nan}

vital = np.array([vitals[df[df["PDAC UniID"]==id]["Vital Status"].array[0]] for id in ids])
survival_months = np.concat([np.array(df[df["PDAC UniID"]==id]["Survival in months"]) for id in ids])
survival_censor = np.concat([np.array(df[df["PDAC UniID"]==id]["Censor for overall survival"]) for id in ids])
recurrence_free = np.concat([np.array(df[df["PDAC UniID"]==id]["Recurrence free in Months"]) for id in ids])
recurrence_censor = np.concat([np.array(df[df["PDAC UniID"]==id]["Censor for recurrence"]) for id in ids])

In [36]:
np.savez("../uni_embs/index_arrays_labeled.npz", 
         dtype = dtype, 
         feat_dim = feat_dim, 
         offsets = offsets, 
         lengths = lengths, 
         slide_ids = ids,
         vital_status = vital,
         survival_months = survival_months,
         survival_censor = survival_censor,
         recurrence_free = recurrence_free,
         recurrence_censor = recurrence_censor,
         total_patches = total_patches)

In [1]:
import os
os.chdir('../')
import numpy as np

from data import *

ind = np.load("../uni_embs/index_arrays_labeled.npz", allow_pickle=True)
valid_inds = [ind for ind, val in enumerate(ind['survival_censor']) if val != np.nan]


ds_index = MemmapDataset(
        data_dir="../uni_embs",
        max_instances=10000,
        indices=valid_inds,
        return_key = True,
        label_column="survival_censor"
    )

ds_index[0]


(tensor([[-0.8066, -0.4124,  0.0482,  ...,  1.1299, -1.3545, -2.0977],
         [-0.9443,  0.2656,  1.2715,  ...,  1.3438, -1.9580, -1.8477],
         [-0.6406,  0.8154,  1.7832,  ...,  1.5732, -1.8066, -1.9561],
         ...,
         [ 0.8535,  0.0264,  0.4419,  ..., -0.3950, -1.1670, -0.1208],
         [-0.0052, -0.4077, -0.9102,  ..., -0.3750,  0.4619,  1.5117],
         [ 0.6606, -1.8438, -1.4395,  ..., -0.6411, -0.8706,  0.3674]],
        dtype=torch.float16),
 np.float64(0.0),
 1)

In [4]:
from torch.utils.data import Dataset, DataLoader

l = DataLoader(
            ds_index,
            batch_size=12,
            shuffle=True,
            num_workers=1,
            pin_memory=False,
            collate_fn=collate_bags,
            drop_last=True,
            persistent_workers=False,
            # REDUCED from 4 to self.prefetch_factor (default 2)
            prefetch_factor=None,
        )

batch = next(iter(l))
view1, labels, keys = batch
len(view1), len(labels), len(keys)

(12, 12, 12)

In [ ]:
import os
os.chdir('../')
import numpy as np

from data import *

ind = np.load("../uni_embs/index_arrays_labeled.npz", allow_pickle=True)
valid_inds = [ind for ind, val in enumerate(ind['survival_censor']) if val != np.nan]


ds_index = MemmapDataset(
        data_dir="../uni_embs",
        max_instances=10000,
        indices=valid_inds,
        return_key = True,
        label_column="survival_censor"
    )

import argparse

import wandb
import numpy as np
from torch import nn

from data import *
from model import EmbMIL

model = EmbMIL()
model = model.to("cuda")

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, betas=(0.9, 0.95))
from torch.optim.lr_scheduler import SequentialLR, CosineAnnealingLR

num_epochs = 5
warmup_epochs = num_epochs // 5
warmup_scheduler = torch.optim.lr_scheduler.LinearLR(optimizer, start_factor=0.01, end_factor=1.0, total_iters=warmup_epochs)
main_scheduler = CosineAnnealingLR(optimizer, T_max=num_epochs - warmup_epochs, eta_min=1e-7)

scheduler = SequentialLR(optimizer, schedulers=[warmup_scheduler, main_scheduler], milestones=[warmup_epochs])

scaler = torch.amp.GradScaler("cuda", enabled=True)

loss_fn = nn.CrossEntropyLoss()

l = DataLoader(
    ds_index,
    batch_size=4,
    shuffle=True,
    num_workers=1,
    pin_memory=False,
    collate_fn=collate_bags,
    drop_last=True,
    persistent_workers=False,
    # REDUCED from 4 to self.prefetch_factor (default 2)
    prefetch_factor=1,
)


for e in range(num_epochs):
    print(f"Epoch {e}: Lr: {scheduler.get_last_lr()}")
    epoch_loss = 0
    epoch_acc = 0
    total_samples = 0
    for batch in l:
        bag, labels, keys = batch
        for sample, label in zip(bag, labels):
            label = torch.tensor([label]).long().to("cuda")
            sample = sample.unsqueeze(0)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=True):
                pred, attn = model.forward_features(sample.to("cuda"))
                loss = loss_fn(pred, label)

            epoch_loss += loss.detach().cpu()
            epoch_acc += np.sum(pred.detach().cpu().numpy().argmax(axis=-1) == label.detach().cpu().int().numpy())

            scaler.scale(loss).backward()

            scaler.step(optimizer)

            # Updates the scale for next iteration.
            scaler.update()

            optimizer.zero_grad() # set_to_none=True here can modestly improve performance
            total_samples += 1
        print(f"Loss: {epoch_loss / total_samples}, Acc: {epoch_acc / total_samples}")
        
    print(f"Loss: {epoch_loss / total_samples}, Acc: {epoch_acc / total_samples}")

    scheduler.step()

Epoch 0: Lr: [1e-05]
Loss: 6.2236328125, Acc: 0.25
Loss: 6.38720703125, Acc: 0.25
Loss: 6.429036617279053, Acc: 0.16666666666666666
Loss: 6.40673828125, Acc: 0.1875
Loss: 6.379687309265137, Acc: 0.2
Loss: 6.33837890625, Acc: 0.16666666666666666
Loss: 6.3311944007873535, Acc: 0.17857142857142858
Loss: 6.3148193359375, Acc: 0.15625
Loss: 6.348849773406982, Acc: 0.1388888888888889
Loss: 6.329785346984863, Acc: 0.15
Loss: 6.330522060394287, Acc: 0.13636363636363635
Loss: 6.333902835845947, Acc: 0.16666666666666666
Loss: 6.332106590270996, Acc: 0.15384615384615385
Loss: 6.325404644012451, Acc: 0.16071428571428573
Loss: 6.303320407867432, Acc: 0.15
Loss: 6.2818603515625, Acc: 0.15625
Loss: 6.283605098724365, Acc: 0.16176470588235295
Loss: 6.275824546813965, Acc: 0.18055555555555555
Loss: 6.276983737945557, Acc: 0.18421052631578946
Loss: 6.287695407867432, Acc: 0.175
Loss: 6.292596817016602, Acc: 0.16666666666666666
Loss: 6.2978515625, Acc: 0.17045454545454544
Loss: 6.292586803436279, Acc: 0.

AcceleratorError: CUDA error: device-side assert triggered
Search for `cudaErrorAssert' in https://docs.nvidia.com/cuda/cuda-runtime-api/group__CUDART__TYPES.html for more information.
CUDA kernel errors might be asynchronously reported at some other API call, so the stacktrace below might be incorrect.
For debugging consider passing CUDA_LAUNCH_BLOCKING=1
Compile with `TORCH_USE_CUDA_DSA` to enable device-side assertions.
